# LangGraph Handoff Demo

Acest notebook demonstrează implementarea unui sistem multi-agent folosind **pattern-ul handoff** cu biblioteca `langgraph` și modelul `llama-3.1-8b-instant` de la Groq. Scenariul simulează un centru de suport clienți în care un agent de **triage (recepție)** preia cererea, o clasifică și **cedează controlul direct** agentului specializat potrivit.

Fluxul de execuție este: `START → triage → [billing | technical] → END`

În **handoff** fiecare agent poate transfera controlul direct unui alt agent, fără a mai trece printr-un coordonator. Agentul `billing` poate face **handoff direct** către `technical` dacă problema depășește domeniul facturării.

Agenții din sistem:
- **`triage`** — clasifică cererea clientului și face handoff la agentul potrivit
- **`billing`** — gestionează întrebări despre facturi și plăți; poate face handoff la `technical`
- **`technical`** — rezolvă probleme tehnice; răspunde direct clientului

Scopul este de a ilustra cum `Command(goto=...)` permite tranziții libere între noduri, fără un nod central de routing.

### Instalarea Bibliotecilor Necesare

Această celulă instalează bibliotecile `langgraph` și `langchain-groq`, esențiale pentru construirea grafului de agenți și pentru interacțiunea cu modelele disponibile prin API-ul Groq. Este o etapă pregătitoare obligatorie înainte de a rula codul principal.

In [1]:
!pip install langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.5 MB/s eta 0:00:00


### Definirea și Execuția Sistemului Multi-Agent cu Handoff (LangGraph)

Această celulă conține implementarea completă a sistemului. Componentele principale sunt:

*   **`GraphState`**: Un `TypedDict` cu câmpurile — `messages: List[BaseMessage]` (istoricul conversației), `current_agent: str` (agentul activ) și `handoff_reason: str` (motivul transferului).
*   **`llm`**: Modelul `llama-3.1-8b-instant` inițializat cu `ChatGroq`, cu `temperature=0` pentru răspunsuri deterministe.
*   **`TriageDecision`**: Model Pydantic pentru **structured output** — triage returnează `next_agent` (`billing` sau `technical`) și `reason`.
*   **`HandoffDecision`**: Model Pydantic pentru decizia specialistului — returnează `needs_handoff` (bool), `handoff_to` (opțional) și `response` (răspunsul scurt).
*   **`triage_node`**: Clasifică cererea și face handoff direct prin `Command(goto=next_agent)`.
*   **`billing_node`**: Gestionează facturi și plăți; poate face handoff la `technical`.
*   **`technical_node`**: Rezolvă probleme tehnice; răspunde direct, fără handoff ulterior.
*   **Construirea grafului**: Punctul de intrare este `triage`; routingul este gestionat exclusiv prin `Command(goto=...)`.
*   **Execuția**: `graph.invoke(...)` pornește cu un `HumanMessage`; triage clasifică și cedează controlul specialistului potrivit.

In [ ]:
from typing import Literal, TypedDict, List, Optional
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from pydantic import BaseModel
from google.colab import userdata

# ──────────────────────────────────────────────
# 1. Modelul LLM
# ──────────────────────────────────────────────
# GROQ_API_KEY poate fi luată accesând link-ul https://console.groq.com/keys
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=userdata.get("GROQ_API_KEY"),
    temperature=0,
)

# ──────────────────────────────────────────────
# 2. Schema de clasificare triage (structured output)
# ──────────────────────────────────────────────
class TriageDecision(BaseModel):
    next_agent: Literal["billing", "technical"]
    reason: str


class HandoffDecision(BaseModel):
    needs_handoff: bool
    handoff_to: Optional[Literal["technical"]] = None
    response: str


llm_triage  = llm.with_structured_output(TriageDecision)
llm_handoff = llm.with_structured_output(HandoffDecision)

TRIAGE_PROMPT = (
    "Triage suport clienți. Clasifică cererea în maxim un cuvânt:\n"
    "  billing → facturi, plăți\n"
    "  technical → erori, conexiune, configurare\n"
    "Alege un singur agent."
)

# ──────────────────────────────────────────────
# 3. Starea grafului
# ──────────────────────────────────────────────
class GraphState(TypedDict):
    messages:       List[BaseMessage]
    current_agent:  str
    handoff_reason: str


# ──────────────────────────────────────────────
# 4. Nodul triage
# ──────────────────────────────────────────────
def triage_node(state: GraphState) -> Command[Literal["billing", "technical"]]:
    decision = llm_triage.invoke(
        [SystemMessage(content=TRIAGE_PROMPT), state["messages"][0]]
    )
    handoff_msg = AIMessage(content=f"[TRIAGE] → {decision.next_agent}: {decision.reason}")
    return Command(
        goto=decision.next_agent,
        update={
            "messages":       state["messages"] + [handoff_msg],
            "current_agent":  decision.next_agent,
            "handoff_reason": decision.reason,
        },
    )


# ──────────────────────────────────────────────
# 5. Noduri specialist
# ──────────────────────────────────────────────
def billing_node(state: GraphState) -> Command:
    system = (
        "Agent billing. Răspunde scurt la întrebări despre facturi și plăți. "
        "Dacă problema e tehnică: needs_handoff=true, handoff_to='technical'. "
        "Răspuns maxim 2 propoziții."
    )
    decision = llm_handoff.invoke(
        [SystemMessage(content=system), state["messages"][0]]
    )
    response_msg = AIMessage(content=f"[BILLING] {decision.response}")

    if decision.needs_handoff and decision.handoff_to:
        return Command(
            goto=decision.handoff_to,
            update={
                "messages":       state["messages"] + [response_msg],
                "current_agent":  decision.handoff_to,
                "handoff_reason": f"Transfer de la billing: {decision.response[:80]}",
            },
        )
    return Command(
        goto=END,
        update={"messages": state["messages"] + [response_msg], "current_agent": "billing"},
    )


def technical_node(state: GraphState) -> Command:
    context = f"\nContext: {state['handoff_reason']}" if state["handoff_reason"] else ""
    system = (
        f"Agent technical. Rezolvă probleme tehnice scurt și clar.{context} "
        "needs_handoff=false. Răspuns maxim 2 propoziții."
    )
    decision = llm_handoff.invoke(
        [SystemMessage(content=system), state["messages"][0]]
    )
    response_msg = AIMessage(content=f"[TECHNICAL] {decision.response}")
    return Command(
        goto=END,
        update={"messages": state["messages"] + [response_msg], "current_agent": "technical"},
    )


# ──────────────────────────────────────────────
# 6. Construirea grafului
# ──────────────────────────────────────────────
builder = StateGraph(GraphState)

builder.add_node("triage",    triage_node)
builder.add_node("billing",   billing_node)
builder.add_node("technical", technical_node)

builder.add_edge(START, "triage")

graph = builder.compile()

# ──────────────────────────────────────────────
# 7. Rulare
# ──────────────────────────────────────────────
result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="Am o taxă necunoscută pe factură din această lună.")
        ],
        "current_agent":  "triage",
        "handoff_reason": "",
    }
)

print("\n=== Istoricul handoff-urilor ===")
for msg in result["messages"]:
    print(f"  {msg.content}")

print("\n=== Răspuns final ===")
print(result["messages"][-1].content)

### Vizualizarea Structurii Grafului

Această celulă generează și afișează o reprezentare vizuală a grafului compilat. Imaginea Mermaid PNG arată nodurile (`triage`, `billing`, `technical`, `sales`) și muchiile dintre ele, ilustrând schema de handoff direct — triage cedează controlul unui specialist, iar specialiștii pot transfera mai departe fără a reveni la un nod central.

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())